# Función de Activación = ReLU, Coeficiente R2, HOLD-Out

In [1]:
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from tqdm import tqdm
import matplotlib.pyplot as plt
from time import time

if torch.cuda.is_available():
    device="cuda"
else:
    device="cpu"
device

'cpu'

In [2]:
data = pd.read_csv('../Shared-the1000/structuralparameters-vs-capacities.csv', index_col=False, header=0)
data

,usablegc,usablevc,density,porosity,Ri,ssa,specific
0,0.015114,0.000241,1.596496,0.007,1.61,211,0.004
1,0.040332,0.000522,1.294051,0.007,1.70,361,0.005
2,0.051645,0.001990,3.851133,0.036,2.02,271,0.009
3,0.052998,0.001876,3.537582,0.014,1.86,232,0.004
4,0.053928,0.000946,1.752954,0.021,1.66,663,0.012
...,...,...,...,...,...,...,...
995,4.350774,0.015634,0.343634,0.675,13.05,4726,1.963
996,4.412086,0.015546,0.336735,0.663,11.24,4885,1.968
997,5.115723,0.015837,0.293674,0.707,8.13,5126,2.406
998,8.004842,0.016169,0.185792,0.790,10.77,5894,4.249


In [7]:
nuevas_dict = { 'VAR6': np.random.randn(1000), 'VAR7': np.random.randn(1000), 'VAR8': np.random.randn(1000)}

In [16]:
for i in range(3):
    data.insert(len(data.columns), column='VAR'+str(6+i), value= np.random.randn(1000))
data

,usablegc,usablevc,density,porosity,Ri,ssa,specific,VAR6,VAR7,VAR8
0,0.015114,0.000241,1.596496,0.007,1.61,211,0.004,0.536650,1.487942,-0.763515
1,0.040332,0.000522,1.294051,0.007,1.70,361,0.005,1.315274,-0.253575,1.464923
2,0.051645,0.001990,3.851133,0.036,2.02,271,0.009,-0.920494,-0.443577,-0.533940
3,0.052998,0.001876,3.537582,0.014,1.86,232,0.004,-0.034148,0.712337,0.583430
4,0.053928,0.000946,1.752954,0.021,1.66,663,0.012,1.426995,-0.062036,0.388746
...,...,...,...,...,...,...,...,...,...,...
995,4.350774,0.015634,0.343634,0.675,13.05,4726,1.963,-1.854035,-0.539060,-1.688515
996,4.412086,0.015546,0.336735,0.663,11.24,4885,1.968,-0.433495,0.964857,-1.342961
997,5.115723,0.015837,0.293674,0.707,8.13,5126,2.406,-1.004949,-0.176314,1.075510
998,8.004842,0.016169,0.185792,0.790,10.77,5894,4.249,-1.771015,0.782336,-0.782071


In [3]:
limites_tabla = pd.read_csv('../Shared-the1000/limits.csv', index_col=False, header=0)
limites_tabla

,Variable,Units,lowlimit,highlimit
0,usablegc,wt.%,0.000001,9.00
1,usablevc,kg/L,0.000001,0.05
2,density,kg/L,0.100000,5.00
3,porosity,dimensionless,0.001000,0.90
4,Ri,angstroms,1.000000,20.00
5,ssa,m^2/g,100.000000,10000.00
6,specific,cm^3/g,0.001000,10.00


In [4]:
limit_inputs = np.array(limites_tabla.iloc[0:2, -2:]).T
limit_inputs

array([[1.e-06, 1.e-06],
       [9.e+00, 5.e-02]])

In [5]:
limit_outputs = np.array(limites_tabla.iloc[2:, -2:]).T
limit_outputs

array([[1.e-01, 1.e-03, 1.e+00, 1.e+02, 1.e-03],
       [5.e+00, 9.e-01, 2.e+01, 1.e+04, 1.e+01]])

In [6]:
escaler_X = MinMaxScaler().fit(limit_inputs)
escaler_y = MinMaxScaler().fit(limit_outputs)

In [7]:
X = torch.tensor(escaler_X.transform(np.array(data.iloc[:, 0:2])), dtype=torch.float32)
X.max(dim=0), X.min(dim=0), X.shape, type(X), X

(torch.return_types.max(
 values=tensor([0.9446, 0.3762]),
 indices=tensor([999, 934])),
 torch.return_types.min(
 values=tensor([0.0017, 0.0048]),
 indices=tensor([0, 0])),
 torch.Size([1000, 2]),
 torch.Tensor,
 tensor([[0.0017, 0.0048],
         [0.0045, 0.0104],
         [0.0057, 0.0398],
         ...,
         [0.5684, 0.3167],
         [0.8894, 0.3234],
         [0.9446, 0.3268]]))

In [8]:
y = torch.tensor(escaler_y.fit_transform(np.array(data.iloc[:, 2:])), dtype=torch.float32)
y.max(dim=0), y.min(dim=0), y.shape, type(y),y

(torch.return_types.max(
 values=tensor([1., 1., 1., 1., 1.]),
 indices=tensor([  2, 999, 995, 984, 999])),
 torch.return_types.min(
 values=tensor([0., 0., 0., 0., 0.]),
 indices=tensor([999,   0, 486, 149,   0])),
 torch.Size([1000, 5]),
 torch.Tensor,
 tensor([[3.8655e-01, 0.0000e+00, 1.0381e-02, 1.6920e-03, 0.0000e+00],
         [3.0425e-01, 0.0000e+00, 1.8166e-02, 2.7073e-02, 2.1763e-04],
         [1.0000e+00, 3.6160e-02, 4.5848e-02, 1.1844e-02, 1.0881e-03],
         ...,
         [3.2066e-02, 8.7282e-01, 5.7439e-01, 8.3333e-01, 5.2274e-01],
         [2.7132e-03, 9.7631e-01, 8.0277e-01, 9.6328e-01, 9.2383e-01],
         [0.0000e+00, 1.0000e+00, 8.2266e-01, 9.4332e-01, 1.0000e+00]]))

In [9]:
# idx_train, idx_test = train_test_split(np.arange(1000), test_size=1/3, random_state=42)

In [10]:
class dataset(Dataset):
  def __init__(self, X: np.ndarray, y: np.ndarray) -> None:
    self.X = X.to(device)
    self.y = y.to(device)
    self.len = self.X.shape[0]
  
  def __getitem__(self, index: int) -> tuple:
    return self.X[index], self.y[index]

  def __len__(self) -> int:
    return self.len

In [17]:
class LinearRegression(nn.Module):
    '''Linear Regression Model
    '''

    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int) -> None:
        '''The network has 4 layers 
             - input layer
             - hidden layer
             - hidden layer
             - output layer
        '''
        super(LinearRegression, self).__init__()
        self.act = torch.nn.ReLU()
        self.input_to_hidden = nn.Linear(input_dim, hidden_dim)
        self.hidden_layer_1 = nn.Linear(hidden_dim, 2*hidden_dim)
        self.hidden_layer_2 = nn.Linear(2*hidden_dim, 2*hidden_dim)

        self.hidden_layer_3 = nn.Linear(2*hidden_dim, hidden_dim)
        self.hidden_to_output = nn.Linear(hidden_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # no activation and no softmax at the end
        x = self.act(self.input_to_hidden(x))
        x = self.act(self.hidden_layer_1(x))
        x = self.act(self.hidden_layer_2(x))
        x = self.act(self.hidden_layer_3(x))
        
        x = self.hidden_to_output(x)
        return x

In [45]:
n = (2+1)*10
n += (10+1)*20
n += (20+1)*20
# n += (20+1)*20
n += (10+1)*8
n

758

In [12]:
CV = KFold(n_splits=10, shuffle=True, random_state=42)

In [33]:
batch_size = 1
input_dim = 2
hidden_layers = 10
output_dim = 8

In [34]:
model = LinearRegression(input_dim, hidden_layers, output_dim)

In [38]:
model

LinearRegression(
  (act): ReLU()
  (input_to_hidden): Linear(in_features=2, out_features=10, bias=True)
  (hidden_layer_1): Linear(in_features=10, out_features=20, bias=True)
  (hidden_layer_2): Linear(in_features=20, out_features=20, bias=True)
  (hidden_layer_3): Linear(in_features=20, out_features=10, bias=True)
  (hidden_to_output): Linear(in_features=10, out_features=8, bias=True)
)

In [35]:
summary(model=model, input_size=(1,2))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                [-1, 1, 10]              30
              ReLU-2                [-1, 1, 10]               0
            Linear-3                [-1, 1, 20]             220
              ReLU-4                [-1, 1, 20]               0
            Linear-5                [-1, 1, 20]             420
              ReLU-6                [-1, 1, 20]               0
            Linear-7                [-1, 1, 10]             210
              ReLU-8                [-1, 1, 10]               0
            Linear-9                 [-1, 1, 8]              88
Total params: 968
Trainable params: 968
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
---------------------------------------------------

In [37]:
inputdatas = 5000*input_dim
inputdatas, inputdatas/968

(10000, 10.330578512396695)

In [9]:
# idx_train, idx_test = train_test_split(np.arange(1000), test_size=1/3, random_state=42)

In [14]:
def training(epochs=10, trainloader=None, X_train=None, y_train=None, X_test=None, y_test=None):
    for epoch in tqdm(range(epochs)):
        model.train()
        for (inputs, labels) in trainloader:
            # forward propagation
            outputs = model(inputs)
            loss = criterion(outputs, labels)
    
            # set optimizer to zero grad 
            # to remove previous epoch gradients
            optimizer.zero_grad()
    
            # backward propagation
            loss.backward()
    
            # optimize
            optimizer.step()
    model.eval()
    with torch.no_grad():  
        y_train_predict = model(X_train)
        y_test_predict = model(X_test)
        loss_train = criterion(y_train_predict, y_train).item()
        loss_test = criterion(y_test_predict, y_test).item()
        R2_train = r2_score(y_train_predict.detach().numpy(), y_train.detach().numpy())
        R2_test = r2_score(y_test_predict.detach().numpy(), y_test.detach().numpy())
        
            
    return loss_train, loss_test, R2_train, R2_test

In [15]:
resultados = pd.DataFrame(columns=['cv', 'train_loss', 'test_loss', 'train_R2', 'test_R2'])

In [16]:
for i , (idx_train, idx_test) in enumerate(CV.split(np.arange(1000))):
    print("CV = ", i)
    
    dataset_train = dataset(X[idx_train], y[idx_train])
    dataset_test = dataset(X[idx_test], y[idx_test])

    trainloader = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)

    
    model = LinearRegression(input_dim, hidden_layers, output_dim).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters())
    rr = training(epochs=200, trainloader=trainloader, X_train=X[idx_train], y_train=y[idx_train], X_test=X[idx_test], y_test=y[idx_test])
    resultados.loc[i] = [i, rr[0], rr[1], rr[2], rr[3]]  

CV =  0


100%|█████████████████████████████████████████| 200/200 [02:49<00:00,  1.18it/s]


CV =  1


100%|█████████████████████████████████████████| 200/200 [02:49<00:00,  1.18it/s]


CV =  2


...

100%|█████████████████████████████████████████| 200/200 [02:49<00:00,  1.18it/s]


CV =  8


100%|█████████████████████████████████████████| 200/200 [02:46<00:00,  1.20it/s]


CV =  9


100%|█████████████████████████████████████████| 200/200 [02:45<00:00,  1.21it/s]


In [17]:
resultados

,cv,train_loss,test_loss,train_R2,test_R2
0,0.0,0.001911,0.001940,0.851485,0.868899
1,1.0,0.001935,0.002151,0.848880,0.836710
2,2.0,0.001819,0.002652,0.882933,0.839985
3,3.0,0.001985,0.001639,0.852215,0.869734
4,4.0,0.001958,0.001566,0.894726,0.877864
5,5.0,0.002178,0.001755,0.833022,0.889951
6,6.0,0.001946,0.002053,0.879079,0.910696
7,7.0,0.001750,0.003314,0.881420,0.737114
8,8.0,0.002295,0.002539,0.837472,0.761592
9,9.0,0.001932,0.001567,0.873284,0.860818


In [18]:
resultados.index = np.arange(10)
resultados

,cv,train_loss,test_loss,train_R2,test_R2
0,0.0,0.001911,0.001940,0.851485,0.868899
1,1.0,0.001935,0.002151,0.848880,0.836710
2,2.0,0.001819,0.002652,0.882933,0.839985
3,3.0,0.001985,0.001639,0.852215,0.869734
4,4.0,0.001958,0.001566,0.894726,0.877864
5,5.0,0.002178,0.001755,0.833022,0.889951
6,6.0,0.001946,0.002053,0.879079,0.910696
7,7.0,0.001750,0.003314,0.881420,0.737114
8,8.0,0.002295,0.002539,0.837472,0.761592
9,9.0,0.001932,0.001567,0.873284,0.860818


In [19]:
resultados.drop(columns='cv', inplace=True)

In [20]:
resultados.index.name='CV'

In [21]:
resultados

,train_loss,test_loss,train_R2,test_R2
CV,,,,
0,0.001911,0.001940,0.851485,0.868899
1,0.001935,0.002151,0.848880,0.836710
2,0.001819,0.002652,0.882933,0.839985
3,0.001985,0.001639,0.852215,0.869734
4,0.001958,0.001566,0.894726,0.877864
5,0.002178,0.001755,0.833022,0.889951
6,0.001946,0.002053,0.879079,0.910696
7,0.001750,0.003314,0.881420,0.737114
8,0.002295,0.002539,0.837472,0.761592


In [22]:
output_model = pd.DataFrame(columns=data.columns)

In [23]:
output_model.loc[0] = [2.75, 0.04, 0, 0, 0, 0, 0]
output_model.loc[1] = [5.5, 0.04, 0, 0, 0, 0, 0]
output_model

,usablegc,usablevc,density,porosity,Ri,ssa,specific
0,2.75,0.04,0.0,0.0,0.0,0.0,0.0
1,5.50,0.04,0.0,0.0,0.0,0.0,0.0


In [24]:
def salida_real(x_real):
    X_real_scaled = escaler_X.transform(x_real)
    y_predict_scaled = model(torch.tensor(X_real_scaled, dtype=torch.float32)).detach().numpy() 
    return escaler_y.inverse_transform(y_predict_scaled)

In [25]:
output_model.iloc[:, 2:] = salida_real(np.array(output_model.iloc[:, :2]))
output_model

,usablegc,usablevc,density,porosity,Ri,ssa,specific
0,2.75,0.04,1.599318,0.469613,6.577447,1442.999268,0.405595
1,5.50,0.04,0.626375,0.771193,10.041493,4115.372559,2.132691
